# Phase 3: Hybrid Exploratory Data Analysis & Base Feature Engineering

## Step 1: Reality Check, Cleansing & Base Feature Translation
A machine learning model cannot find a statistical edge in a dataset that is too small, and it cannot mathematically process string objects (like timestamps such as "08:30:00"). 

Before running advanced correlation metrics, i have clean the data and transform the text objects into usable numerical features. I will apply the following pipeline across the datasets:

1. **Initial Size Check:** Any dataset with fewer than 500 instances is immediately discarded to prevent overfitting (Data Starvation).
2. **Data Cleansing:** Drop any rows with missing values (`NaN`) and remove exact duplicate rows.
3. **Secondary Size Check:** If the cleansing process drops the dataset below 500 instances, the file is deleted.
4. **Base Feature Engineering (Object Translation):** I will parse non-numeric object columns (like `Time` and `Date`) into raw numerical formats (e.g., `Hour`, `Minute`, `Day_of_Week`). *Note: No data scaling or normalization occurs at this stage.*
5. **Advanced EDA (The Audit):** With all features successfully converted to numbers, i can accurately check feature distributions, target baselines, and correlation matrices to finalize the datasets.

In [1]:
import os
import glob
import pandas as pd


data_dir = '../data/processed/' 
csv_files = glob.glob(os.path.join(data_dir, '*.csv'))

MIN_INSTANCES = 500
deleted_files = 0
kept_files = 0



for file in csv_files:
    filename = os.path.basename(file)
    df = pd.read_csv(file)
    
    # 1. Initial size check
    if len(df) < MIN_INSTANCES:
        
        os.remove(file)
        deleted_files += 1
        continue
        
    # 2. Clean data (Drop NaNs and Duplicates)
    original_len = len(df)
    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)
    cleaned_len = len(df)
    
    # 3. Secondary size check after cleaning
    if cleaned_len < MIN_INSTANCES:
        
        os.remove(file)
        deleted_files += 1
    else:
        # 4. Save cleaned data back to disk
        df.to_csv(file, index=False)
        dropped_rows = original_len - cleaned_len
        
        kept_files += 1


print(f"{kept_files} files survived. {deleted_files} files deleted.")

60 files survived. 20 files deleted.


## Step 2: Pattern-Specific EDA (Quality Over Quantity)

Because each ICT pattern has a completely unique schema (e.g., FVG has `Gap_Size`, Sweep has `Sweep_Volume`), i can no longer process them in a single batch. I have to isolate each pattern and audit its specific features. 

my core philosophy for this phase is strict: **5 working, highly predictive models are infinitely better than 60 broken, noisy ones.** If a dataset cannot pass the following rigorous checks, it will be permanently deleted.

For each pattern, i will execute the following audit:
1. **Target Analysis & Dummy Baseline:** Checking the `TP_or_SL` ratio. If a dataset has a 90% loss rate, it is statistically dead. I will establish the "Dummy Baseline" (the natural win rate) that the ML model must beat.
2. **Feature Distribution & Noise Removal:** Hunting for extreme outliers (e.g., a data glitch creating a 10,000-pip gap). i will drop rows containing extreme noise.
3. **Correlation & Multicollinearity:** Checking if features are perfectly correlated with each other (which confuses models) or highly correlated with the target. 



### 2A. Fair Value Gap (FVG) Exploratory Data Analysis
**Focus:** Analyzing the relationship between the 3-candle body sizes, volume concentration, and the gap size against the final trade outcome. i will check for extreme gap anomalies and establish the baseline win rate across the surviving FVG datasets.

In [4]:
import os
import glob
import pandas as pd
import numpy as np


csv_files = glob.glob(os.path.join('../data/processed/', 'fvg_*.csv'))

MIN_INSTANCES = 500
LOWER_WIN_RATE = 0.10
UPPER_WIN_RATE = 0.90
CORR_THRESHOLD = 0.85


outlier_cols = [
    'C1_Body', 'C2_Body', 'C3_Body', 
    '3C_Volume', 'Gap_Size', 
    'Entry_Body', 'Entry_Volume','TP_Size'
]

summary_log = []

for file in csv_files:
    filename = os.path.basename(file)
    df = pd.read_csv(file)
    

        
    
    win_rate = df['TP_or_SL'].mean()
    if win_rate < LOWER_WIN_RATE or win_rate > UPPER_WIN_RATE:
        summary_log.append(f" {filename:<20} Deleted: Extreme Win Rate ({win_rate:.1%})")
        os.remove(file)
        continue
        
    
    df['Dataset_WinRate'] = win_rate
        
    
    initial_len = len(df)
    for col in outlier_cols:
        if col in df.columns:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
            
    
    final_len = len(df)
    if final_len < MIN_INSTANCES:
        summary_log.append(f"{filename:<20} Deleted: Size dropped to {final_len} after removing outliers")
        os.remove(file)
        continue
        
    
    features_to_check = df.drop(columns=['TP_or_SL', 'Dataset_WinRate'], errors='ignore')
    corr_matrix = features_to_check.corr().abs()
    
    
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    
    dropped_features = {}
    for col in upper.columns:
        high_corr = upper[col][upper[col] > CORR_THRESHOLD]
        if not high_corr.empty:
            correlated_with = high_corr.idxmax() 
            dropped_features[col] = correlated_with
            
    
    if dropped_features:
        df.drop(columns=list(dropped_features.keys()), inplace=True)
        
    
    df.to_csv(file, index=False)
    
    
    corr_msg = ""
    if dropped_features:
        corr_msg = " | Dropped Multicollinearity: " + ", ".join([f"{k} (due to {v})" for k, v in dropped_features.items()])
        
    summary_log.append(f" {filename:<20} Kept: {final_len} rows, Win Rate: {win_rate:.1%}{corr_msg}")


print("FVG EDA AUDIT REPORT")
print("="*90)
for log in sorted(summary_log):
    print(log)

FVG EDA AUDIT REPORT
 fvg_15M_EURUSD.csv   Kept: 10028 rows, Win Rate: 34.2%
 fvg_15M_Gold.csv     Kept: 9987 rows, Win Rate: 35.4%
 fvg_15M_Nasdaq.csv   Kept: 5204 rows, Win Rate: 36.0%
 fvg_15M_SP500.csv    Kept: 8546 rows, Win Rate: 36.0%
 fvg_15M_Silver.csv   Kept: 9891 rows, Win Rate: 34.8%
 fvg_1H_EURUSD.csv    Kept: 2311 rows, Win Rate: 36.0%
 fvg_1H_Gold.csv      Kept: 2108 rows, Win Rate: 37.2%
 fvg_1H_Nasdaq.csv    Kept: 1254 rows, Win Rate: 39.6%
 fvg_1H_SP500.csv     Kept: 1984 rows, Win Rate: 38.1%
 fvg_1H_Silver.csv    Kept: 2247 rows, Win Rate: 35.6%
 fvg_5M_EURUSD.csv    Kept: 32646 rows, Win Rate: 33.7%
 fvg_5M_Gold.csv      Kept: 33001 rows, Win Rate: 34.4%
 fvg_5M_Nasdaq.csv    Kept: 16878 rows, Win Rate: 35.6%
 fvg_5M_SP500.csv     Kept: 25817 rows, Win Rate: 35.0%
 fvg_5M_Silver.csv    Kept: 31513 rows, Win Rate: 34.0%


### 2B. Order Block (OB) Exploratory Data Analysis
**Focus:** Order Blocks rely heavily on displacement and volatility. i will audit the `ATR_at_Formation` and the displacement volume (`C3_Volume` vs `C1/C2`) to remove extreme volatility outliers and check for multicollinearity between the volume nodes.

In [5]:
import os
import glob
import pandas as pd
import numpy as np


csv_files = glob.glob(os.path.join('../data/processed/', 'ob_*.csv'))

MIN_INSTANCES = 500
LOWER_WIN_RATE = 0.10
UPPER_WIN_RATE = 0.90
CORR_THRESHOLD = 0.85


outlier_cols = [
    'C1_Volume', 'C2_Volume', 'C3_Volume', 
    'ATR_at_formation', 'TP_Size'
]

summary_log = []

for file in csv_files:
    filename = os.path.basename(file)
    df = pd.read_csv(file)
    
    
        
    
    win_rate = df['TP_or_SL'].mean()
    if win_rate < LOWER_WIN_RATE or win_rate > UPPER_WIN_RATE:
        summary_log.append(f" {filename:<20} Deleted: Extreme Win Rate ({win_rate:.1%})")
        os.remove(file)
        continue
        
    
    df['Dataset_WinRate'] = win_rate
        
    
    initial_len = len(df)
    for col in outlier_cols:
        if col in df.columns:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
            
    
    final_len = len(df)
    if final_len < MIN_INSTANCES:
        summary_log.append(f" {filename:<20} Deleted: Size dropped to {final_len} after removing outliers")
        os.remove(file)
        continue
        
    
    features_to_check = df.drop(columns=['TP_or_SL', 'Dataset_WinRate'], errors='ignore')
    corr_matrix = features_to_check.corr().abs()
    
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    dropped_features = {}
    for col in upper.columns:
        high_corr = upper[col][upper[col] > CORR_THRESHOLD]
        if not high_corr.empty:
            dropped_features[col] = high_corr.idxmax()
            
    if dropped_features:
        df.drop(columns=list(dropped_features.keys()), inplace=True)
        
    
    df.to_csv(file, index=False)
    
    corr_msg = ""
    if dropped_features:
        corr_msg = " | Dropped Multicollinearity: " + ", ".join([f"{k} (due to {v})" for k, v in dropped_features.items()])
        
    summary_log.append(f" {filename:<20} Kept: {final_len} rows, Win Rate: {win_rate:.1%}{corr_msg}")


print("ORDER BLOCK EDA AUDIT REPORT")
print("="*90)
for log in sorted(summary_log):
    print(log)

ORDER BLOCK EDA AUDIT REPORT
 ob_15M_EURUSD.csv    Kept: 6857 rows, Win Rate: 35.0% | Dropped Multicollinearity: C2_Volume (due to C1_Volume), Formation_Session (due to Formation_Hour)
 ob_15M_Gold.csv      Kept: 6480 rows, Win Rate: 36.6% | Dropped Multicollinearity: Formation_Session (due to Formation_Hour)
 ob_15M_Nasdaq.csv    Kept: 3781 rows, Win Rate: 37.9% | Dropped Multicollinearity: C2_Volume (due to C1_Volume), C3_Volume (due to C2_Volume), Formation_Session (due to Formation_Hour)
 ob_15M_SP500.csv     Kept: 5435 rows, Win Rate: 36.5% | Dropped Multicollinearity: Formation_Session (due to Formation_Hour)
 ob_15M_Silver.csv    Kept: 6201 rows, Win Rate: 36.8% | Dropped Multicollinearity: Formation_Session (due to Formation_Hour)
 ob_1H_EURUSD.csv     Kept: 1817 rows, Win Rate: 34.9% | Dropped Multicollinearity: Formation_Session (due to Formation_Hour)
 ob_1H_Gold.csv       Kept: 1598 rows, Win Rate: 37.5% | Dropped Multicollinearity: Formation_Session (due to Formation_Hour)

### 2C. Liquidity Sweep Exploratory Data Analysis
**Focus:** Sweeps are built on time-anchored liquidity pools. i will audit `Sweep_Volume` and `Gap_Size` (the distance between the high/low pool). i have to rigorously check the win/loss distribution here, as sweep reversals often have naturally lower win rates but higher R:R.

In [6]:
import os
import glob
import pandas as pd
import numpy as np

csv_files = glob.glob(os.path.join('../data/processed/', 'sweep_*.csv'))

MIN_INSTANCES = 500
LOWER_WIN_RATE = 0.10
UPPER_WIN_RATE = 0.90
CORR_THRESHOLD = 0.85

outlier_cols = [
    'Gap_Size', 'Sweep_Volume', 'TP_Size'
]

summary_log = []

for file in csv_files:
    filename = os.path.basename(file)
    df = pd.read_csv(file)
    
    if df.empty:
        continue
        
    win_rate = df['TP_or_SL'].mean()
    if win_rate < LOWER_WIN_RATE or win_rate > UPPER_WIN_RATE:
        summary_log.append(f"[DROPPED] {filename:<20} Deleted: Extreme Win Rate ({win_rate:.1%})")
        os.remove(file)
        continue
        
    df['Dataset_WinRate'] = win_rate
        
    initial_len = len(df)
    for col in outlier_cols:
        if col in df.columns:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
            
    final_len = len(df)
    if final_len < MIN_INSTANCES:
        summary_log.append(f"[DROPPED] {filename:<20} Deleted: Size dropped to {final_len} after removing outliers")
        os.remove(file)
        continue
        
    features_to_check = df.drop(columns=['TP_or_SL', 'Dataset_WinRate'], errors='ignore')
    corr_matrix = features_to_check.corr().abs()
    
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    dropped_features = {}
    for col in upper.columns:
        high_corr = upper[col][upper[col] > CORR_THRESHOLD]
        if not high_corr.empty:
            dropped_features[col] = high_corr.idxmax()
            
    if dropped_features:
        df.drop(columns=list(dropped_features.keys()), inplace=True)
        
    df.to_csv(file, index=False)
    
    corr_msg = ""
    if dropped_features:
        corr_msg = " | Dropped Multicollinearity: " + ", ".join([f"{k} (due to {v})" for k, v in dropped_features.items()])
        
    summary_log.append(f"[KEPT]    {filename:<20} Kept: {final_len} rows, Win Rate: {win_rate:.1%}{corr_msg}")

print("LIQUIDITY SWEEP EDA AUDIT REPORT")
print("="*90)
for log in sorted(summary_log):
    print(log)

LIQUIDITY SWEEP EDA AUDIT REPORT
[DROPPED] sweep_15M_SP500.csv  Deleted: Extreme Win Rate (9.4%)
[DROPPED] sweep_1H_Nasdaq.csv  Deleted: Size dropped to 431 after removing outliers
[DROPPED] sweep_5M_EURUSD.csv  Deleted: Extreme Win Rate (8.9%)
[DROPPED] sweep_5M_Gold.csv    Deleted: Extreme Win Rate (9.5%)
[DROPPED] sweep_5M_SP500.csv   Deleted: Extreme Win Rate (8.1%)
[DROPPED] sweep_5M_Silver.csv  Deleted: Extreme Win Rate (8.5%)
[KEPT]    sweep_15M_EURUSD.csv Kept: 3637 rows, Win Rate: 10.8% | Dropped Multicollinearity: Lows_DayOfWeek (due to Highs_DayOfWeek), Sweep_DayOfWeek (due to Highs_DayOfWeek), Highs_Session (due to Highs_Hour), Lows_Session (due to Lows_Hour), Sweep_Session (due to Sweep_Hour)
[KEPT]    sweep_15M_Gold.csv   Kept: 3400 rows, Win Rate: 10.2% | Dropped Multicollinearity: Lows_DayOfWeek (due to Highs_DayOfWeek), Sweep_DayOfWeek (due to Highs_DayOfWeek), Highs_Session (due to Highs_Hour), Lows_Session (due to Lows_Hour), Sweep_Session (due to Sweep_Hour)
[KEPT] 

### 2D. Optimal Trade Entry (OTE) Exploratory Data Analysis
**Focus:** OTE is heavily dependent on the size of the impulse leg (`Gap_Size`). i will filter out micro-impulses that are too small to trade and audit the entry candle characteristics to ensure no future bias leaked into the Fibonacci retracement calculations.

In [7]:
import os
import glob
import pandas as pd
import numpy as np

csv_files = glob.glob(os.path.join('../data/processed/', 'ote_*.csv'))

MIN_INSTANCES = 500
LOWER_WIN_RATE = 0.10
UPPER_WIN_RATE = 0.90
CORR_THRESHOLD = 0.85

outlier_cols = [
    'Gap_Size', 'Entry_Volume', 'Entry_Body', 'TP_Size'
]

summary_log = []

for file in csv_files:
    filename = os.path.basename(file)
    df = pd.read_csv(file)
    
    if df.empty:
        continue
        
    win_rate = df['TP_or_SL'].mean()
    if win_rate < LOWER_WIN_RATE or win_rate > UPPER_WIN_RATE:
        summary_log.append(f"[DROPPED] {filename:<20} Deleted: Extreme Win Rate ({win_rate:.1%})")
        os.remove(file)
        continue
        
    df['Dataset_WinRate'] = win_rate
        
    initial_len = len(df)
    for col in outlier_cols:
        if col in df.columns:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
            
    final_len = len(df)
    if final_len < MIN_INSTANCES:
        summary_log.append(f"[DROPPED] {filename:<20} Deleted: Size dropped to {final_len} after removing outliers")
        os.remove(file)
        continue
        
    features_to_check = df.drop(columns=['TP_or_SL', 'Dataset_WinRate'], errors='ignore')
    corr_matrix = features_to_check.corr().abs()
    
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    dropped_features = {}
    for col in upper.columns:
        high_corr = upper[col][upper[col] > CORR_THRESHOLD]
        if not high_corr.empty:
            dropped_features[col] = high_corr.idxmax()
            
    if dropped_features:
        df.drop(columns=list(dropped_features.keys()), inplace=True)
        
    df.to_csv(file, index=False)
    
    corr_msg = ""
    if dropped_features:
        corr_msg = " | Dropped Multicollinearity: " + ", ".join([f"{k} (due to {v})" for k, v in dropped_features.items()])
        
    summary_log.append(f"[KEPT]    {filename:<20} Kept: {final_len} rows, Win Rate: {win_rate:.1%}{corr_msg}")

print("OPTIMAL TRADE ENTRY (OTE) EDA AUDIT REPORT")
print("="*90)
for log in sorted(summary_log):
    print(log)

OPTIMAL TRADE ENTRY (OTE) EDA AUDIT REPORT
[KEPT]    ote_15M_EURUSD.csv   Kept: 21682 rows, Win Rate: 31.9% | Dropped Multicollinearity: TP_Size (due to Gap_Size), Lows_DayOfWeek (due to Highs_DayOfWeek), Entry_DayOfWeek (due to Highs_DayOfWeek), Highs_Session (due to Highs_Hour), Lows_Session (due to Lows_Hour), Entry_Session (due to Entry_Hour)
[KEPT]    ote_15M_Gold.csv     Kept: 21372 rows, Win Rate: 32.2% | Dropped Multicollinearity: TP_Size (due to Gap_Size), Lows_DayOfWeek (due to Highs_DayOfWeek), Entry_DayOfWeek (due to Highs_DayOfWeek), Highs_Session (due to Highs_Hour), Lows_Session (due to Lows_Hour), Entry_Session (due to Entry_Hour)
[KEPT]    ote_15M_Nasdaq.csv   Kept: 10960 rows, Win Rate: 28.6% | Dropped Multicollinearity: TP_Size (due to Gap_Size), Lows_DayOfWeek (due to Highs_DayOfWeek), Entry_DayOfWeek (due to Highs_DayOfWeek), Highs_Session (due to Highs_Hour), Lows_Session (due to Lows_Hour), Entry_Session (due to Entry_Hour)
[KEPT]    ote_15M_SP500.csv    Kept: 18

## Step 3: Feature Scaling, Cyclical Encoding & The Final Purge (The Hybrid Finish)

While scaling is traditionally part of the Feature Engineering phase, this Hybrid Architecture allows me to finish the data preparation right here. By keeping it in this notebook, I ensure a seamless transition from outlier removal directly into model-ready datasets. 

### The Liquidity Sweep Purge
Before scaling, I made a crucial decision: **The complete removal of the Liquidity Sweep strategy.** 
The baseline win rates for Sweeps hovered around 10%. If I fed this to a machine learning model, the severe class imbalance would cause the algorithm to simply predict "Stop Loss" on every single trade. It would achieve 90% accuracy without learning a single structural feature, rendering the model completely useless. I have deleted all `sweep_*.csv` files to keep the pipeline focused only on statistically viable setups (FVG, OB, and OTE).

### Feature Scaling & Cyclical Encoding for Neural Networks
Tree-based models (like Random Forest or XGBoost) do not require feature scaling, but Multi-Layer Perceptrons (MLPs) and distance-based classifiers do. To ensure the datasets are compatible with any algorithm, I will apply two specific transformations:

*   **Continuous Features (Standard Normalization):** Scaled using Z-score (Mean = 0, Variance = 1) so large values don't artificially dominate the network's gradient descent.
*   **Time Features (Cyclical Encoding):** Leaving `Hour` (0-23) and `DayOfWeek` (0-6) as raw integers creates a magnitude problem for MLPs, and standard scaling destroys their cyclical nature (the fact that Hour 23 is right next to Hour 0). Instead, I will transform these into Sine and Cosine waves. This naturally bounds them between -1.0 and 1.0 while teaching the model that time is a continuous circle.
*   **Target Feature:** `TP_or_SL` remains completely untouched (strictly 0 or 1).

The scalers will be saved to a master `../scalers/` directory for use in live trading, and the final cleansed, scaled datasets will be exported to `../data/model_ready/`.

In [9]:
import os
import glob
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler


processed_dir = '../data/processed/'
ready_dir = '../data/model_ready/'
scaler_dir = '../scalers/'


os.makedirs(ready_dir, exist_ok=True)
os.makedirs(scaler_dir, exist_ok=True)

csv_files = glob.glob(os.path.join(processed_dir, '*.csv'))

print("Starting Step 3: Feature Scaling & Cyclical Encoding...\n" + "="*70)

for file in csv_files:
    filename = os.path.basename(file)
       
    df = pd.read_csv(file)
    

    hour_cols = [col for col in df.columns if col.endswith('_Hour')]
    day_cols = [col for col in df.columns if col.endswith('_DayOfWeek')]
    
    
    for col in hour_cols:
        df[col + '_sin'] = np.sin(2 * np.pi * df[col] / 24.0)
        df[col + '_cos'] = np.cos(2 * np.pi * df[col] / 24.0)
        df.drop(columns=[col], inplace=True) 
        
    
    for col in day_cols:
        df[col + '_sin'] = np.sin(2 * np.pi * df[col] / 7.0)
        df[col + '_cos'] = np.cos(2 * np.pi * df[col] / 7.0)
        df.drop(columns=[col], inplace=True) 
        
    
    cols_to_scale = [
        col for col in df.columns 
        if col not in ['TP_or_SL', 'Dataset_WinRate'] 
        and not col.endswith('_sin') 
        and not col.endswith('_cos')
    ]
    
    if cols_to_scale:
        scaler = StandardScaler()
        df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
        
        
        scaler_filename = filename.replace('.csv', '_scaler.pkl')
        joblib.dump(scaler, os.path.join(scaler_dir, scaler_filename))
    
    
    ready_filepath = os.path.join(ready_dir, filename)
    df.to_csv(ready_filepath, index=False)
    
    print(f"Encoded, Scaled, and Exported: {filename}")

print("="*70)
print(f"EDA Phase Complete. All model-ready data saved to {ready_dir}")
print(f"All scaler objects saved to {scaler_dir}")

Starting Step 3: Feature Scaling & Cyclical Encoding...
Encoded, Scaled, and Exported: fvg_15M_EURUSD.csv
Encoded, Scaled, and Exported: fvg_15M_Gold.csv
Encoded, Scaled, and Exported: fvg_15M_Nasdaq.csv
Encoded, Scaled, and Exported: fvg_15M_Silver.csv
Encoded, Scaled, and Exported: fvg_15M_SP500.csv
Encoded, Scaled, and Exported: fvg_1H_EURUSD.csv
Encoded, Scaled, and Exported: fvg_1H_Gold.csv
Encoded, Scaled, and Exported: fvg_1H_Nasdaq.csv
Encoded, Scaled, and Exported: fvg_1H_Silver.csv
Encoded, Scaled, and Exported: fvg_1H_SP500.csv
Encoded, Scaled, and Exported: fvg_5M_EURUSD.csv
Encoded, Scaled, and Exported: fvg_5M_Gold.csv
Encoded, Scaled, and Exported: fvg_5M_Nasdaq.csv
Encoded, Scaled, and Exported: fvg_5M_Silver.csv
Encoded, Scaled, and Exported: fvg_5M_SP500.csv
Encoded, Scaled, and Exported: ob_15M_EURUSD.csv
Encoded, Scaled, and Exported: ob_15M_Gold.csv
Encoded, Scaled, and Exported: ob_15M_Nasdaq.csv
Encoded, Scaled, and Exported: ob_15M_Silver.csv
Encoded, Scaled, an